# Spectral Graph Analysis for Coupling Networks

Spectral methods reveal synchronization structure from the coupling
matrix alone, before running any dynamics.

**Key tools:**

- **Critical coupling K_c** -- the minimum effective coupling for
  frequency synchronization. From the graph Laplacian: 
  `K_c = max|omega_i - omega_j| / lambda_2(L)` 
  (Dorfler & Bullo 2014, Automatica 50(6):1539-1564).

- **Fiedler partitioning** -- the sign of the Fiedler vector (eigenvector
  of lambda_2) bisects the network into two synchronization clusters.

- **Lyapunov guard** -- tracks `V(theta) = -(K/2N) sum K_ij cos(theta_i - theta_j)`.
  Kuramoto dynamics are gradient flow on V, so V(t) is non-increasing
  (van Hemmen & Wreszinski 1993, J. Stat. Phys. 72:145-166).

- **Normalized Persistent Entropy (NPE)** -- topological sync metric
  from H0 persistence. NPE ~ 0 = synchronized, NPE ~ 1 = incoherent.
  More sensitive than R for detecting partial synchronization.

In [ ]:
import numpy as np

from scpn_phase_orchestrator.coupling.spectral import (
    critical_coupling,
    fiedler_partition,
    fiedler_value,
    fiedler_vector,
    graph_laplacian,
    spectral_gap,
    sync_convergence_rate,
)
from scpn_phase_orchestrator.monitor.lyapunov import LyapunovGuard
from scpn_phase_orchestrator.monitor.npe import compute_npe
from scpn_phase_orchestrator.upde.engine import UPDEEngine
from scpn_phase_orchestrator.upde.order_params import compute_order_parameter

In [ ]:
# Build coupling matrix: 8 oscillators with exponential decay coupling
N_OSC = 8
BASE_K = 0.6
DECAY = 0.4

idx = np.arange(N_OSC)
dist = np.abs(idx[:, np.newaxis] - idx[np.newaxis, :])
knm = BASE_K * np.exp(-DECAY * dist)
np.fill_diagonal(knm, 0.0)

omegas = np.array([0.8, 0.9, 1.0, 1.1, 0.85, 0.95, 1.05, 1.15])

# Spectral analysis
L = graph_laplacian(knm)
lam2 = fiedler_value(knm)
K_c = critical_coupling(omegas, knm)
gap = spectral_gap(knm)
conv_rate = sync_convergence_rate(knm, omegas)

print(f"Lambda_2 (algebraic connectivity): {lam2:.4f}")
print(f"Critical coupling K_c:             {K_c:.4f}")
print(f"Spectral gap (lam3 - lam2):        {gap:.4f}")
print(f"Estimated convergence rate:         {conv_rate:.4f}")
print(f"\nK_effective / K_c = {BASE_K / K_c:.2f} (>1 means sync is expected)")

In [ ]:
# Fiedler partition: bisect the network into sync clusters
v2 = fiedler_vector(knm)
group_pos, group_neg = fiedler_partition(knm)

print("Fiedler vector:")
for i, val in enumerate(v2):
    group = "+" if val >= 0 else "-"
    print(f"  osc {i}: v2={val:+.4f}  omega={omegas[i]:.2f}  [{group}]")

print(f"\nGroup (+): {group_pos}")
print(f"Group (-): {group_neg}")

try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Coupling matrix heatmap
    im = axes[0].imshow(knm, cmap="viridis")
    axes[0].set_title("Coupling matrix K_ij")
    axes[0].set_xlabel("j")
    axes[0].set_ylabel("i")
    plt.colorbar(im, ax=axes[0], fraction=0.046)

    # Fiedler vector bar chart
    colors = ["tab:blue" if v >= 0 else "tab:red" for v in v2]
    axes[1].bar(range(N_OSC), v2, color=colors)
    axes[1].axhline(0, color="black", linewidth=0.5)
    axes[1].set_xlabel("Oscillator")
    axes[1].set_ylabel("v2 component")
    axes[1].set_title("Fiedler Vector (sync partition)")

    fig.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping plot")

In [ ]:
# Run Lyapunov guard: show V(t) decreasing during Kuramoto dynamics
DT = 0.01
STEPS = 2000
alpha = np.zeros((N_OSC, N_OSC))

engine = UPDEEngine(N_OSC, dt=DT)
guard = LyapunovGuard()

rng = np.random.default_rng(42)
phases = rng.uniform(0, 2 * np.pi, N_OSC)

V_hist, dV_hist, basin_hist, R_hist = [], [], [], []
for _ in range(STEPS):
    phases = engine.step(phases, omegas, knm, 0.0, 0.0, alpha)
    ls = guard.evaluate(phases, knm)
    V_hist.append(ls.V)
    dV_hist.append(ls.dV_dt)
    basin_hist.append(ls.in_basin)
    R, _ = compute_order_parameter(phases)
    R_hist.append(R)

print(f"V(0)={V_hist[0]:.4f}  V(end)={V_hist[-1]:.4f}")
print(f"V decreased: {V_hist[-1] < V_hist[0]}")
print(f"In basin at end: {basin_hist[-1]}")

try:
    import matplotlib.pyplot as plt

    t = np.arange(STEPS) * DT
    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

    axes[0].plot(t, V_hist, linewidth=0.8, color="tab:purple")
    axes[0].set_ylabel("V(t) (Lyapunov)")
    axes[0].set_title("Lyapunov Function Descent")

    axes[1].plot(t, R_hist, linewidth=0.8, color="tab:blue", label="R")
    axes[1].set_ylabel("R")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylim(-0.05, 1.1)
    axes[1].legend()

    fig.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping plot")

In [ ]:
# Compute NPE and compare with R over time
# Re-run simulation recording both metrics
engine2 = UPDEEngine(N_OSC, dt=DT)
guard2 = LyapunovGuard()
rng2 = np.random.default_rng(42)
phases2 = rng2.uniform(0, 2 * np.pi, N_OSC)

R_track, NPE_track = [], []
# Sample every 10 steps to keep NPE computation reasonable
SAMPLE_EVERY = 10

for step in range(STEPS):
    phases2 = engine2.step(phases2, omegas, knm, 0.0, 0.0, alpha)
    if step % SAMPLE_EVERY == 0:
        R, _ = compute_order_parameter(phases2)
        npe = compute_npe(phases2)
        R_track.append(R)
        NPE_track.append(npe)

print(f"Final R:   {R_track[-1]:.4f}")
print(f"Final NPE: {NPE_track[-1]:.4f}")
print("Correlation: R ~ 1 corresponds to NPE ~ 0 (both indicate sync)")

try:
    import matplotlib.pyplot as plt

    t_sampled = np.arange(len(R_track)) * SAMPLE_EVERY * DT
    fig, ax1 = plt.subplots(figsize=(10, 4))

    color_r = "tab:blue"
    ax1.plot(t_sampled, R_track, color=color_r, linewidth=1.0, label="R")
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("R (order parameter)", color=color_r)
    ax1.tick_params(axis="y", labelcolor=color_r)
    ax1.set_ylim(-0.05, 1.1)

    ax2 = ax1.twinx()
    color_npe = "tab:orange"
    ax2.plot(t_sampled, NPE_track, color=color_npe, linewidth=1.0, label="NPE")
    ax2.set_ylabel("NPE", color=color_npe)
    ax2.tick_params(axis="y", labelcolor=color_npe)
    ax2.set_ylim(-0.05, 1.1)

    fig.legend(loc="upper right", bbox_to_anchor=(0.95, 0.95), fontsize=9)
    ax1.set_title("R vs NPE: Complementary Sync Metrics")
    fig.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping plot")